# RAG Hand-On

*Run the following script to prepare for this pipeline:*

`pip install -U langchain langchain-community langchain-huggingface langchain-text-splitters faiss-cpu unstructured[pdf] sentence-transformers  langchain_groq`

## Supply data for RAG chat bot
Loading internal documentation or a corpus and setting up MARKDOWN_SEPARATORS to supply a database for the AI. This step helps our chatbot respond based on the supplied documentation.

In [4]:
from langchain_community.document_loaders import DirectoryLoader, UnstructuredFileLoader
loader = DirectoryLoader(
    path="./Corpus",
    glob= "**/*.pdf",
    loader_cls=UnstructuredFileLoader,
    show_progress= True,
    use_multithreading= True
)

MARKDOWN_SEPARATORS = [
    "\n\n",   # ưu tiên tách theo đoạn
    "\n",     # rồi tới dòng
    ". ",     # rồi tới câu
    " ",      # rồi tới từ
    ""        # cuối cùng tách ký tự
]


docs = loader.load()


c:\Users\Lenovo\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/2 [00:00<?, ?it/s]

Need to load profiles.
100%|██████████| 2/2 [00:04<00:00,  2.43s/it]


# Chunking Step

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1200,
    chunk_overlap = 200,
    add_start_index = True,
    strip_whitespace = True,
    separators= MARKDOWN_SEPARATORS
)

In [6]:
from pprint import pprint
splits =text_splitter.split_documents(docs)
pprint(splits)

[Document(metadata={'source': 'Corpus\\under_earth_volume2_bestiary_settlements.pdf', 'start_index': 0}, page_content='Under-Earth Survival Guide\n\nVolume II — Bestiary, Settlements, and Travel Routes\n\nPrepared as fictional source material for retrieval-augmented generation practice.\n\nEdition 1.0 · English · Entirely fictional\n\nUnder-Earth Survival Guide · Volume II\n\n1. Civilized Life Below\n\nSurvival becomes culture when people stay long enough to name winds, argue over maps, and\n\ncreate rules around danger. The settlements below are adapted to biology, darkness, and trade\n\nrather than agriculture under sun.\n\nMost settlements are built near predictable water and low-spore air, not near the largest food\n\nsources.\n\n\n\nTrade value is dominated by salt, dry fungus, lamp oils, woven silk, medicinal resins, and\n\naccurate maps.\n\nReputation matters more than currency on frontier routes. A guide known for truthful\n\nwarnings is worth more than polished gear.\n\n2. Maj

# Embedding model

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {"device": "cpu"}
encode_kwargs = {"normalize_embeddings": False}

embedding_model = HuggingFaceEmbeddings(
    model_name = model_name,
    model_kwargs = model_kwargs,
    encode_kwargs = encode_kwargs   
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5851.97it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Test embedding model

In [8]:
from langchain_core.documents import Document

docs = [
    Document(page_content="RAG combines retrieval and generation."),
    Document(page_content="Embeddings turn text into vectors."),
    Document(page_content="FAISS can store vectors for similarity search."),
    Document(page_content="Dog is the viewer, who look at this line. ")
]

for doc in docs:
    embedded_doc =  embedding_model.embed_query(doc.page_content)
    print(embedded_doc)


[0.010324089787900448, 0.05444912239909172, -0.015484404750168324, -0.005214184056967497, -0.013607271946966648, 0.00377609021961689, 0.006058442406356335, 0.014005766250193119, -0.07641521841287613, 0.013866429217159748, 0.0073149921372532845, -0.04069025069475174, -0.014421064406633377, -0.02472565695643425, 0.026924606412649155, -0.025394145399332047, 0.025787685066461563, 0.016472328454256058, -0.07839927822351456, -0.02491423487663269, -0.005541256163269281, -0.016892092302441597, 0.004180125426501036, -0.004521673079580069, -0.016939695924520493, 0.01427080761641264, -0.013578851707279682, 0.03895222023129463, -0.014774642884731293, -0.01049187034368515, -0.01913452334702015, 0.008349175564944744, 0.019338561221957207, 0.021097013726830482, 1.541216761324904e-06, -0.040730856359004974, -0.02792959287762642, -0.014367341995239258, -0.020313598215579987, 0.033144887536764145, 0.10073521733283997, 0.07380769401788712, -0.038488756865262985, 0.005503259599208832, -0.00669932737946510

# Vector database

In [9]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy

vector_store = FAISS.from_documents(
    embedding=embedding_model,
    documents=splits, 
    distance_strategy=DistanceStrategy.COSINE
)


# Kiểm tra tổng số lượng documents đã được index
total_docs = len(vector_store.docstore._dict)
print(f"Số lượng chunks hiện có: {total_docs}")

# Nếu bạn muốn xem thử nội dung của 1 mẩu đầu tiên
first_key = list(vector_store.docstore._dict.keys())[0]
print(f"Nội dung mẩu đầu tiên: {vector_store.docstore._dict[first_key].page_content}")

Số lượng chunks hiện có: 16
Nội dung mẩu đầu tiên: Under-Earth Survival Guide

Volume II — Bestiary, Settlements, and Travel Routes

Prepared as fictional source material for retrieval-augmented generation practice.

Edition 1.0 · English · Entirely fictional

Under-Earth Survival Guide · Volume II

1. Civilized Life Below

Survival becomes culture when people stay long enough to name winds, argue over maps, and

create rules around danger. The settlements below are adapted to biology, darkness, and trade

rather than agriculture under sun.

Most settlements are built near predictable water and low-spore air, not near the largest food

sources.



Trade value is dominated by salt, dry fungus, lamp oils, woven silk, medicinal resins, and

accurate maps.

Reputation matters more than currency on frontier routes. A guide known for truthful

warnings is worth more than polished gear.

2. Major Settlements of Under-Earth

Settlement

Built Around

Known For

Social Rule

Hearthwake

A ring 

Save vector database to pkl file:

In [10]:
vector_store.save_local("Vector_Store")

In [11]:
vector_store = FAISS.load_local(
    "Vector_Store",
    embedding_model,
    allow_dangerous_deserialization=True
)

retriever = vector_store.as_retriever(
    search_type = "similarity_score_threshold",
    search_kwargs= {"k": 5,"score_threshold": 0.2}
)

query = "What is Under-Earth"
# Lấy ra các docs liên quan nhất
docs_related = retriever.invoke(query)

if not docs_related:
    print("Không tìm thấy tài liệu nào vượt qua ngưỡng 0.1 rồi!")
else:
    for i, doc in enumerate(docs_related):
        print(f"\nKết quả {i+1}:")
        print(doc.page_content)


Kết quả 1:
Under-Earth Survival Guide

Volume I — Field Manual for First Descent Teams

Prepared as fictional source material for retrieval-augmented generation practice.

Edition 1.0 · English · Entirely fictional

Under-Earth Survival Guide · Volume I

1. Orientation

Under-Earth is a hypothetical biosphere beneath the continental crust: humid, dim, vast, and

alive. New arrivals do not survive by strength alone. They survive by rhythm, caution, and

respect for the ecology.



The ceiling is rarely stable. Never camp beneath fresh glass-vine growth, fractured mineral

ribs, or hanging fungal shelves.

Most native light is biological rather than solar. A bright area is not automatically safe; many

luminous organisms attract scavengers.

Running water is valuable but dangerous. Streams often collect spores, metallic salts, or

larvae in seasonal pulses.



Sound travels unpredictably in hollow stone. A whisper may carry farther than a shout in

flute-tunnels and resonance wells.

Pr

# LLM

Set the rule for RAG

In [12]:
template_prompt = """
You are a retrieval-augmented QA assistant.

RULES:
- Answer only from the provided context.
- Do not use outside knowledge.
- If the answer is not in the context, say: "I don't know based on the provided context."
- Do not make up information.
- Keep the answer clear and concise.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
"""

In [13]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
key = os.getenv("GROQ_API_KEY")
load_dotenv()
print("KEY FOUND:", key is not None)
print("KEY PREFIX:", key[:8] if key else None)
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=os.getenv("GROQ_API_KEY")
)

KEY FOUND: True
KEY PREFIX: gsk_A9uW


In [14]:
def ask_rag(question: str):
    docs_related = retriever.invoke(question)

    if not docs_related:
        return "I don't know based on the provided context."

    context = "\n\n".join(doc.page_content for doc in docs_related)

    prompt = template_prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(prompt)

    return response.content

In [15]:
query = "What is Under-Earth?"
answer = ask_rag(query)


print("Question:", query)
print("Answer:", answer)

Question: What is Under-Earth?
Answer: Under-Earth is a hypothetical biosphere beneath the continental crust: humid, dim, vast, and alive.
